In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


In [ ]:
#fazer uma correlacao entre links com mesma origem para mebasar o motivo de termos juntado
import os
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, mannwhitneyu

def analyze_datasets_without_timestamp(path, substring):
    # Listar arquivos no diretório com o substring
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    results = []  # Armazenar resultados das comparações
    
    # Comparação entre datasets
    for i, df1 in enumerate(dataframe_list):
        for j, df2 in enumerate(dataframe_list):
            if i >= j:
                continue
            
            # Verificar colunas idênticas em ordem e nome
            if set(df1.columns) != set(df2.columns):
                print(f"Datasets {i+1} e {j+1} não possuem as mesmas colunas. Ignorando comparação.")
                continue
            
            shared_columns = list(set(df1.columns) & set(df2.columns))
            comparison_result = {
                "Dataset1": f"Dataset {i+1}",
                "Dataset2": f"Dataset {j+1}",
                "Column_Comparisons": []
            }
            
            for col in shared_columns:
                if pd.api.types.is_numeric_dtype(df1[col]) and pd.api.types.is_numeric_dtype(df2[col]):
                    # Estatísticas básicas
                    stats1 = df1[col].describe()
                    stats2 = df2[col].describe()
                    
                    # Teste KS para distribuições
                    ks_stat, ks_pvalue = ks_2samp(df1[col].dropna(), df2[col].dropna())
                    
                    # Teste Mann-Whitney U
                    mw_stat, mw_pvalue = mannwhitneyu(df1[col].dropna(), df2[col].dropna(), alternative='two-sided')
                    
                    # Armazenar resultados
                    comparison_result["Column_Comparisons"].append({
                        "Column": col,
                        "Dataset1_Stats": stats1.to_dict(),
                        "Dataset2_Stats": stats2.to_dict(),
                        "Kolmogorov-Smirnov": {"Statistic": ks_stat, "P-Value": ks_pvalue},
                        "Mann-Whitney-U": {"Statistic": mw_stat, "P-Value": mw_pvalue}
                    })
            
            results.append(comparison_result)
    
    # Exibir resultados
    for result in results:
        print(f"\nComparação entre {result['Dataset1']} e {result['Dataset2']}:")
        for comp in result["Column_Comparisons"]:
            print(f"  Coluna: {comp['Column']}")
            print(f"    Estatísticas {result['Dataset1']}: {comp['Dataset1_Stats']}")
            print(f"    Estatísticas {result['Dataset2']}: {comp['Dataset2_Stats']}")
            print(f"    Kolmogorov-Smirnov: {comp['Kolmogorov-Smirnov']}")
            print(f"    Mann-Whitney-U: {comp['Mann-Whitney-U']}")
    
    return results


path = "../../datasets\serie-multivariada"
substring = "ma"
analyze_datasets_without_timestamp(path, substring)


In [ ]:
# Imputação - BBR
bbr = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
columns_to_drop = ['PolynomialRegression', 'AdaBoostRegressor', 'ElasticNet', 
                   'LinearRegression', 'MLPRegressor', 'SVR', 'KNeighborsRegressor']
sources_to_drop = ['df', 'rj', 'sp', 'pa', 'sc', 'pr', 'mg']

bbr = bbr.drop(columns=columns_to_drop, errors='ignore')
bbr = bbr[~bbr['source'].isin(sources_to_drop)]

models = bbr.columns[1:]
y = np.arange(len(bbr['source']))  # Agora as categorias estão no eixo y
height = 0.7 / len(models)         # Ajustando altura das barras para maior largura

custom_colors = [
    'blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'maroon', 'brown'

]

# Ajustando o tamanho da figura para ser menor em largura e maior em comprimento
fig, ax = plt.subplots(figsize=(10, 12))  

for i, model in enumerate(models):
    # Usando as cores definidas manualmente
    ax.barh(y + i * height, bbr[model], height, label=model, color=custom_colors[i], edgecolor='black')  # Adicionando borda preta nas barras

ax.set_title("Comparação de RMSE por modelo de predição - BBR", fontsize=14)
ax.set_xlabel("NRMSE", fontsize=12)  # Eixo x agora representa os valores
ax.set_ylabel("Source", fontsize=12)  # Eixo y representa as categorias
ax.set_yticks(y + height * (len(models) / 2 - 0.5))
ax.set_yticklabels(bbr['source'])

# Ajustando o espaço abaixo do gráfico para a legenda
plt.subplots_adjust(bottom=0.15)

# Legenda ajustada para ficar mais próxima do gráfico
ax.legend(
    title="Modelos de Predição",
    bbox_to_anchor=(0.5, -0.05),  # Ajusta a posição para ficar mais próxima
    loc='upper center',
    ncol=3
)

ax.grid(axis='x', linestyle='--', alpha=0.7)  # Grid no eixo x
plt.tight_layout()
plt.show()



# # Imputação - CUBIC
# cubic = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_cubic_rmse.csv')
# models = cubic.columns[1:]
# y = np.arange(len(cubic['source']))
# height = 0.75 / len(models)
# colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

# fig, ax = plt.subplots(figsize=(12, 8))
# for i, model in enumerate(models):
#     ax.barh(y + i * height, cubic[model], height, label=model, color=colors[i])

# ax.set_title("Comparação de RMSE por modelo de predição - CUBIC", fontsize=14)
# ax.set_xlabel("NRMSE", fontsize=12)
# ax.set_ylabel("Source", fontsize=12)
# ax.set_yticks(y + height * (len(models) / 2 - 0.5))
# ax.set_yticklabels(cubic['source'])

# # Legenda ajustada para baixo
# ax.legend(
#     title="Modelos de Predição",
#     bbox_to_anchor=(0.5, -0.2),
#     loc='upper center',
#     ncol=3
# )

# ax.grid(axis='x', linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.show()

# Métricas estatísticas:
#                             nrmse                                
#                              mean     std  median     min     max
# model                                                            
# AdaBoostRegressor          0.2676  0.3231  0.0591  0.0359  0.9362
# CatBoostRegressor          0.2224  0.2615  0.0511  0.0314  0.7054
# ElasticNet                 0.2934  0.3607  0.0609  0.0379  1.0186
# GradientBoostingRegressor  0.2310  0.2735  0.0512  0.0315  0.7379
# KNeighborsRegressor        0.2470  0.2870  0.0570  0.0358  0.7750
# LGBMRegressor              0.2241  0.2627  0.0513  0.0318  0.7195
# LinearRegression           0.2934  0.3607  0.0609  0.0379  1.0186
# MLPRegressor               0.3439  0.3611  0.0924  0.0379  1.0381
# PolynomialRegression       0.2711  0.3258  0.0560  0.0360  0.9310
# RandomForestRegressor      0.2275  0.2671  0.0518  0.0317  0.7224
# SVR                        0.3548  0.4437  0.0657  0.0399  1.2578
# XGBRegressor               0.2285  0.2701  0.0520  0.0316  0.7304

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

def plot_enhanced_scatter(df_model, save_path=None):
    # Set the style
    sns.set_style('whitegrid')  # Use a Seaborn style (e.g., 'whitegrid', 'darkgrid')
    sns.set_palette("husl")
    
    # Create figure and axis with specified size
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create scatter plot with larger points and alpha
    scatter = ax.scatter(df_model['RMSE'], 
                        df_model['Variance_coef'],
                        s=100,  # Larger points
                        alpha=0.6,  # Some transparency
                        c=df_model['RMSE'],  # Color by RMSE
                        cmap='viridis')
    
    # Add colorbar
    plt.colorbar(scatter, label='RMSE Value')
    
    # Calculate and plot regression line
    x = df_model['RMSE']
    y = df_model['Variance_coef']
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    line = slope * x + intercept
    ax.plot(x, line, color='red', linestyle='--', 
            label=f'R² = {r_value**2:.3f}')
    
    # Add confidence interval
    plt.fill_between(x, 
                     line - std_err, 
                     line + std_err, 
                     alpha=0.2, 
                     color='red')
    
    # Customize the plot
    ax.set_title('Relationship between RMSE and Coefficient of Variance',
                fontsize=16, pad=20)
    ax.set_xlabel('Normalized RMSE', fontsize=12)
    ax.set_ylabel('Coefficient of Variance', fontsize=12)
    
    # Add grid with lower opacity
    ax.grid(True, alpha=0.3)
    
    # Add legend
    ax.legend(fontsize=10)
    
    # Tight layout
    plt.tight_layout()
    
    # Save if path is provided
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
    return fig, ax


def plot_joint_distribution(df_model, save_path=None):
    # Create joint plot
    g = sns.jointplot(data=df_model,
                      x='RMSE',
                      y='Variance_coef',
                      kind='reg',  # Include regression line
                      height=10,
                      ratio=8,
                      marginal_kws=dict(bins=20),
                      joint_kws={'scatter_kws': {'alpha': 0.5}})
    
    # Customize the plot
    g.fig.suptitle('RMSE vs Coefficient of Variance Distribution', 
                   y=1.02, fontsize=16)
    g.ax_joint.set_xlabel('Normalized RMSE', fontsize=12)
    g.ax_joint.set_ylabel('Coefficient of Variance', fontsize=12)
    
    # Add correlation coefficient
    corr = df_model['RMSE'].corr(df_model['Variance_coef'])
    g.ax_joint.text(0.05, 0.95, f'Correlation: {corr:.3f}',
                    transform=g.ax_joint.transAxes,
                    fontsize=10)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        
    return g

# Example usage:
fig, ax = plot_enhanced_scatter(df_model, 'scatter_plot.png')
g = plot_joint_distribution(df_model, 'joint_distribution.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Lendo os arquivos CSV
df1 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_bbr.csv')
df2 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_cubic.csv')

# Adicionando sufixos diferentes à coluna 'Source' para diferenciar as fontes
df1['Source'] = df1['Source'].astype(str) + 'BBR'  # Sufixo para df1
df2['Source'] = df2['Source'].astype(str) + 'CUBIC'  # Sufixo para df2

# Unindo os dois dataframes (um abaixo do outro)
df_concatenado = pd.concat([df1, df2], ignore_index=True)


# Reorganizando os dados para formato longo (melhor para plotagem)
rmse_columns = [col for col in df_concatenado.columns if col.startswith('RMSE')]
df_melted = pd.melt(df_concatenado, 
                    id_vars=['Source', 'Variance_coef'],
                    value_vars=rmse_columns,
                    var_name='Model',
                    value_name='RMSE')

# Limpando os nomes dos modelos (removendo o prefixo 'RMSE_')
df_melted['Model'] = df_melted['Model'].str.replace('RMSE_', '')

# Configurando o estilo do plot
plt.style.use('ggplot')  # Estilo válido como alternativa ao 'seaborn'
plt.figure(figsize=(15, 10))

# Criando o scatter plot
sns.scatterplot(data=df_melted,
                x='Variance_coef',
                y='RMSE',
                hue='Model',
                style='Model',
                s=100,  # tamanho dos pontos
                alpha=0.7)  # transparência

# Adicionando linha de tendência
z = np.polyfit(df_melted['Variance_coef'], df_melted['RMSE'], 1)
p = np.poly1d(z)
plt.plot(df_melted['Variance_coef'], p(df_melted['Variance_coef']), 
         "r--", alpha=0.8, label='Trend Line')

# Customizando o plot
plt.title('Relationship between Variance Coefficient and RMSE for Different Models',
          fontsize=16, pad=20)
plt.xlabel('Coefficient of Variance', fontsize=12)
plt.ylabel('RMSE', fontsize=12)

# Ajustando a legenda
plt.legend(bbox_to_anchor=(1.05, 1), 
           loc='upper left', 
           borderaxespad=0.,
           fontsize=10)

# Ajustando os limites dos eixos para melhor visualização
plt.xlim(-5, max(df_melted['Variance_coef']) * 1.1)
plt.ylim(-0.05, max(df_melted['RMSE']) * 1.1)

# Adicionando grid
plt.grid(True, alpha=0.3)

# Ajustando o layout para não cortar a legenda
plt.tight_layout()

# Salvando o gráfico
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')

# Mostrando o gráfico
plt.show()

# Calculando e mostrando correlações
correlations = []
for model in df_melted['Model'].unique():
    model_data = df_melted[df_melted['Model'] == model]
    corr = np.corrcoef(model_data['Variance_coef'], model_data['RMSE'])[0,1]
    correlations.append({
        'Model': model,
        'Correlation': corr
    })

correlations_df = pd.DataFrame(correlations)
correlations_df = correlations_df.sort_values('Correlation', ascending=False)
print("\nCorrelações entre Coefficient of Variance e RMSE por modelo:")
print(correlations_df)


In [ ]:
#vendo qual melhor modelo de acordo com o nrmse - RMSE_CatBoostRegressor
df1 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_bbr.csv')
df2 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_cubic.csv')

df1['Source'] = df1['Source'].astype(str) + 'BBR'  
df2['Source'] = df2['Source'].astype(str) + 'CUBIC'  
df_concatenado = pd.concat([df1, df2], ignore_index=True)
#removendo as colunas que nao quero calcular a media
df_without_first_two = df_concatenado.iloc[:, 2:]

mean_values = df_without_first_two.mean()

print(mean_values)
# melhor eh catboostregressor

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Lendo o CSV
df = df_concatenado

# 1. Preparando os dados
rmse_columns = [col for col in df.columns if col.startswith('RMSE')]
df_melted = pd.melt(df, 
                    id_vars=['Source', 'Variance_coef'],
                    value_vars=rmse_columns,
                    var_name='Model',
                    value_name='RMSE')

# Certifique-se de que 'RMSE' é numérico
df_melted['RMSE'] = pd.to_numeric(df_melted['RMSE'], errors='coerce')  # Erros serão convertidos para NaN

# Garantir que 'Source' seja uma coluna de strings
df_melted['Source'] = df_melted['Source'].astype(str)

# Remover valores NaN antes de continuar
df_melted = df_melted.dropna(subset=['RMSE'])

# Configurando o estilo
plt.style.use('ggplot')  # Mudado para estilo válido 'ggplot'

# 1. Boxplot comparando performance dos modelos
plt.figure(figsize=(15, 6))
sns.boxplot(data=df_melted, x='Model', y='RMSE')
plt.xticks(rotation=45, ha='right')
plt.title('Distribution of RMSE by Model')
plt.tight_layout()
plt.savefig('1_model_comparison_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Heatmap das médias de RMSE por estado
plt.figure(figsize=(15, 8))
rmse_pivot = df_melted.pivot(index='Source', columns='Model', values='RMSE')
sns.heatmap(rmse_pivot, cmap='YlOrRd', annot=True, fmt='.3f', cbar_kws={'label': 'RMSE'})
plt.title('RMSE Values by State and Model')
plt.tight_layout()
plt.savefig('2_rmse_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Barplot com os melhores modelos por estado
best_models = df_melted.loc[df_melted.groupby('Source')['RMSE'].idxmin()]
plt.figure(figsize=(15, 6))
sns.barplot(data=best_models, x='Source', y='RMSE', hue='Model')
plt.xticks(rotation=45, ha='right')
plt.title('Best Performing Model by State')
plt.tight_layout()
plt.savefig('3_best_models.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. Violin plot da distribuição de RMSE
plt.figure(figsize=(15, 6))
sns.violinplot(data=df_melted, x='Model', y='RMSE')
plt.xticks(rotation=45, ha='right')
plt.title('RMSE Distribution by Model')
plt.tight_layout()
plt.savefig('4_rmse_distribution.png', dpi=300, bbox_inches='tight')
plt.close()

# 5. Calculando e plotando médias e desvios padrão
model_stats = df_melted.groupby('Model')['RMSE'].agg(['mean', 'std']).sort_values('mean')
plt.figure(figsize=(12, 6))
model_stats.plot(kind='bar', yerr='std', capsize=5)
plt.title('Mean RMSE with Standard Deviation by Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('5_model_stats.png', dpi=300, bbox_inches='tight')
plt.close()

# Gerando tabela com estatísticas resumidas
stats_summary = pd.DataFrame({
    'Mean RMSE': df_melted.groupby('Model')['RMSE'].mean(),
    'Std RMSE': df_melted.groupby('Model')['RMSE'].std(),
    'Min RMSE': df_melted.groupby('Model')['RMSE'].min(),
    'Max RMSE': df_melted.groupby('Model')['RMSE'].max(),
    # 'Best States': df_melted.loc[df_melted.groupby('Model')['RMSE'].idxmin()]['Source'],
    # 'Worst States': df_melted.loc[df_melted.groupby('Model')['RMSE'].idxmax()]['Source']
}).round(4)

print("\nModel Performance Summary:")
print(stats_summary)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

def categorize_nrmse(value):
    if value < 0.1:
        return 'Excelente'
    elif value < 0.2:
        return 'Bom'
    elif value < 0.3:
        return 'Razoável'
    elif value < 0.5:
        return 'Ruim'
    else:
        return 'Muito Ruim'

def analyze_nrmse(df):
    df_melted = df.melt(id_vars=['source'], var_name='model', value_name='nrmse')
    
    # Adicionar categorias
    df_melted['categoria'] = df_melted['nrmse'].apply(categorize_nrmse)
    
    # 1. Análise geral por modelo
    model_analysis = df_melted.groupby('model').agg({
        'nrmse': ['mean', 'std', 'median', 'min', 'max']
    }).round(4)
    
    # 2. Distribuição das categorias por modelo
    category_dist = pd.crosstab(df_melted['model'], df_melted['categoria'], normalize='index') * 100
    
    # 3. Identificar melhores estados por modelo
    best_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmin()]
    
    # 4. Identificar piores estados por modelo
    worst_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmax()]
    
    return df_melted, model_analysis, category_dist, best_states, worst_states

def plot_analysis(df_melted, model_analysis, category_dist):
    """
    Cria visualizações para a análise do NRMSE.
    """
    plt.style.use('default')
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Boxplot dos modelos
    plt.subplot(2, 2, 1)
    sns.boxplot(data=df_melted, x='model', y='nrmse', width=0.7)
    plt.xticks(rotation=45, ha='right')
    plt.title('Distribuição do NRMSE por Modelo')
    plt.xlabel('Modelo')
    plt.ylabel('NRMSE')
    
    # 2. Heatmap da distribuição de categorias
    plt.subplot(2, 2, 2)
    sns.heatmap(category_dist, annot=True, fmt='.1f', cmap='YlOrRd')
    plt.title('Distribuição das Categorias por Modelo (%)')
    plt.xlabel('Categoria')
    plt.ylabel('Modelo')
    
    # 3. Gráfico de barras do NRMSE médio
    plt.subplot(2, 2, 3)
    model_means = model_analysis['nrmse']['mean'].sort_values()
    plt.bar(range(len(model_means)), model_means)
    plt.xticks(range(len(model_means)), model_means.index, rotation=45, ha='right')
    plt.title('NRMSE Médio por Modelo')
    plt.xlabel('Modelo')
    plt.ylabel('NRMSE Médio')
    
    # 4. Gráfico de dispersão do NRMSE por estado
    plt.subplot(2, 2, 4)
    markers = ['o', 's', '^', 'v', 'D', 'p', 'h', '8', '*', '+', 'x', 'd']
    for i, model in enumerate(df_melted['model'].unique()):
        model_data = df_melted[df_melted['model'] == model]
        plt.scatter(model_data['source'], model_data['nrmse'], 
                   label=model, alpha=0.6, marker=markers[i % len(markers)])
    plt.xticks(rotation=45, ha='right')
    plt.title('NRMSE por Estado e Modelo')
    plt.xlabel('Estado')
    plt.ylabel('NRMSE')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    return fig


df = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')


df_melted, model_analysis, category_dist, best_states, worst_states = analyze_nrmse(df)

fig = plot_analysis(df_melted, model_analysis, category_dist)

# Imprimir resultados detalhados
print("\n=== Análise por Modelo ===")
print("\nMétricas estatísticas:")
print(model_analysis)

print("\n=== Melhores Estados por Modelo ===")
best_states_formatted = best_states[['model', 'source', 'nrmse']].sort_values('nrmse')
print(best_states_formatted.to_string())

print("\n=== Piores Estados por Modelo ===")
worst_states_formatted = worst_states[['model', 'source', 'nrmse']].sort_values('nrmse', ascending=False)
print(worst_states_formatted.to_string())

# Calcular e mostrar distribuição geral das categorias
total_dist = df_melted['categoria'].value_counts(normalize=True) * 100
print("\n=== Distribuição Geral das Categorias ===")
print(total_dist.round(2).sort_index())

# Salvar resultados em arquivos
output_dir = '../../results/regression/analysis'
os.makedirs(output_dir, exist_ok=True)

model_analysis.to_csv(f'{output_dir}/model_analysis_bbr.csv')
category_dist.to_csv(f'{output_dir}/category_distribution_bbr.csv')
plt.savefig(f'{output_dir}/nrmse_analysis_plots_bbr.png', bbox_inches='tight', dpi=300)

# Salvar também um resumo em formato mais legível
with open(f'{output_dir}/analysis_summary_bbr.txt', 'w') as f:
    f.write("=== Análise de NRMSE ===\n\n")
    f.write("Distribuição das Categorias:\n")
    f.write(total_dist.round(2).sort_index().to_string())
    f.write("\n\nMelhores Estados:\n")
    f.write(best_states_formatted.to_string())
    f.write("\n\nPiores Estados:\n")
    f.write(worst_states_formatted.to_string())